### Imports

In [1]:
import json
import numpy as np
import pandas as pd
import pingouin as pg
import seaborn as sn

print(pg.__version__) # 0.5.3
print(pd.__version__) # 2.0.3
print(np.__version__) # 1.24.3
print(sn.__version__) # 0.13.0

from utils_MS import *

# %load_ext autotime

0.5.3
2.2.2
1.26.4
0.13.2


/home/ealvarez/miniconda3/envs/graph_matching/lib/python3.10/site-packages/outdated/utils.py:14: OutdatedPackageWarning: The package pingouin is out of date. Your version is 0.5.3, the latest is 0.6.0.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(


In [2]:
# def run(exp):

### Parameters

In [3]:
file = open("exp.json")
experiment = json.load(file)
exp = experiment["exp"]

file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

data_variations = params["data_variations"]
print("Data variations:", data_variations)

has_transformation = params["has_transformation"]
print("Has transformation:", has_transformation)

threshold_corr = params["threshold_corr"]
print("Threshold corr:\t", threshold_corr)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

groups_id_no = params["groups_id_no"]
print("Groups id (no):\t", groups_id_no)

Exp:		 exp6
Data variations: ['none']
Has transformation: False
Threshold corr:	 0.5
Groups id:	 ['SecoAmazonas', 'SecoCusco', 'SecoSanMartin', 'FrescoAmazonas', 'FrescoCusco', 'FrescoSanMartin']
Subgroups id:	 {'SecoAmazonas': ['1', '2'], 'SecoCusco': ['1', '2'], 'SecoSanMartin': ['1', '2'], 'FrescoAmazonas': ['1', '2'], 'FrescoCusco': ['1', '2'], 'FrescoSanMartin': ['1', '2']}
Groups id (no):	 ['Blank', 'QC', 'Std']


### Load dataset

In [4]:
# read raw data
df_join_raw = pd.read_csv("experiments/input/{}_raw.csv".format(exp), index_col=0)
df_join_raw

,Average Rt,Average Mz,Metabolite name,SecoAmazonas_1.1,SecoAmazonas_1.2,SecoAmazonas_1.3,SecoAmazonas_2.1,SecoAmazonas_2.2,SecoAmazonas_2.3,SecoCusco_1.1,...,FrescoCusco_1.3,FrescoCusco_2.1,FrescoCusco_2.2,FrescoCusco_2.3,FrescoSanMartin_1.1,FrescoSanMartin_1.2,FrescoSanMartin_1.3,FrescoSanMartin_2.1,FrescoSanMartin_2.2,FrescoSanMartin_2.3
0,2.120,152.05702,unknown,1.947976e+06,2.145146e+06,1.889418e+06,1.756070e+06,1.281853e+06,1.737510e+06,2.758043e+06,...,2.788924e+05,2.495039e+05,4.248775e+05,3.098800e+05,2.888458e+05,4.263454e+05,2.601564e+05,2.230069e+05,2.389870e+05,1.698564e+05
1,2.125,257.96816,unknown,1.241109e+07,2.402011e+06,2.862010e+06,2.414489e+06,3.688394e+06,4.478066e+06,4.357772e+06,...,1.291696e+06,4.211488e+06,4.924348e+06,1.276892e+07,1.048620e+07,4.959735e+06,1.233451e+07,5.116804e+06,8.661161e+05,1.246542e+07
2,2.128,207.98572,unknown,5.125119e+06,6.072819e+06,5.915750e+06,7.670063e+06,1.112191e+07,6.413633e+06,5.885871e+06,...,6.876271e+06,5.189219e+06,5.727825e+06,3.114297e+06,4.682256e+06,1.237203e+07,5.174974e+06,6.716172e+06,7.540170e+06,7.938045e+06
3,2.133,217.96001,unknown,1.784870e+06,1.576395e+06,9.802001e+05,1.714031e+06,1.990814e+06,2.014875e+06,9.986963e+05,...,1.598919e+06,1.272286e+06,1.024954e+06,1.517614e+06,1.434988e+06,1.253071e+06,1.796903e+06,1.959247e+06,1.377944e+06,1.642902e+06
4,2.255,152.05702,unknown,1.947976e+06,2.145146e+06,1.889418e+06,1.756070e+06,1.281853e+06,1.737510e+06,2.758043e+06,...,2.788924e+05,2.495039e+05,4.248775e+05,3.098800e+05,2.888458e+05,4.263454e+05,2.601564e+05,2.230069e+05,2.389870e+05,1.698564e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,13.576,359.19705,unknown,2.946200e+07,3.335107e+07,3.520060e+07,2.942578e+07,3.004647e+07,3.151865e+07,3.869054e+07,...,3.547148e+07,3.244064e+07,3.301073e+07,3.187038e+07,3.838530e+07,2.989014e+07,3.527329e+07,2.920353e+07,3.044284e+07,3.695492e+07
78,13.782,515.23988,unknown,3.533964e+06,4.744814e+06,3.217944e+06,3.319008e+06,4.928576e+06,3.432516e+06,5.062720e+06,...,2.779232e+06,3.833238e+06,4.436233e+06,4.582209e+06,4.422106e+06,3.285745e+06,4.329191e+06,3.609709e+06,4.725286e+06,3.467679e+06
79,14.886,667.23171,unknown,2.096873e+06,2.334722e+06,2.275303e+06,1.700928e+06,2.053808e+06,2.442797e+06,2.437916e+06,...,2.267375e+06,2.866012e+06,2.191374e+06,1.940903e+06,2.732096e+06,1.846165e+06,2.074487e+06,2.443682e+06,1.844960e+06,3.324116e+06
80,15.129,599.31616,unknown,1.193444e+06,4.534499e+06,5.824158e+06,4.594740e+06,8.490425e+05,3.976399e+06,4.301930e+06,...,4.308660e+06,1.364190e+06,1.602308e+06,3.487020e+06,3.755398e+06,4.040052e+06,1.644414e+06,3.098998e+06,2.836007e+06,1.862349e+06


In [5]:
# get metadata
df_join_raw_metadata = df_join_raw.iloc[:, :2]
df_join_raw_metadata

,Average Rt,Average Mz
0,2.120,152.05702
1,2.125,257.96816
2,2.128,207.98572
3,2.133,217.96001
4,2.255,152.05702
...,...,...
77,13.576,359.19705
78,13.782,515.23988
79,14.886,667.23171
80,15.129,599.31616


In [6]:
# filter by samples
columns_sample = [column for column in df_join_raw.columns if column.split("_")[0] not in groups_id_no]
df_join_raw_intensity = df_join_raw.loc[:, columns_sample]
df_join_raw_intensity = df_join_raw_intensity.iloc[:, 3:]
df_join_raw_intensity

,SecoAmazonas_1.1,SecoAmazonas_1.2,SecoAmazonas_1.3,SecoAmazonas_2.1,SecoAmazonas_2.2,SecoAmazonas_2.3,SecoCusco_1.1,SecoCusco_1.2,SecoCusco_1.3,SecoCusco_2.1,...,FrescoCusco_1.3,FrescoCusco_2.1,FrescoCusco_2.2,FrescoCusco_2.3,FrescoSanMartin_1.1,FrescoSanMartin_1.2,FrescoSanMartin_1.3,FrescoSanMartin_2.1,FrescoSanMartin_2.2,FrescoSanMartin_2.3
0,1.947976e+06,2.145146e+06,1.889418e+06,1.756070e+06,1.281853e+06,1.737510e+06,2.758043e+06,2.721496e+06,2.639917e+06,2.665687e+06,...,2.788924e+05,2.495039e+05,4.248775e+05,3.098800e+05,2.888458e+05,4.263454e+05,2.601564e+05,2.230069e+05,2.389870e+05,1.698564e+05
1,1.241109e+07,2.402011e+06,2.862010e+06,2.414489e+06,3.688394e+06,4.478066e+06,4.357772e+06,3.483170e+06,3.947152e+06,5.173156e+06,...,1.291696e+06,4.211488e+06,4.924348e+06,1.276892e+07,1.048620e+07,4.959735e+06,1.233451e+07,5.116804e+06,8.661161e+05,1.246542e+07
2,5.125119e+06,6.072819e+06,5.915750e+06,7.670063e+06,1.112191e+07,6.413633e+06,5.885871e+06,5.947384e+06,8.102998e+06,7.490531e+06,...,6.876271e+06,5.189219e+06,5.727825e+06,3.114297e+06,4.682256e+06,1.237203e+07,5.174974e+06,6.716172e+06,7.540170e+06,7.938045e+06
3,1.784870e+06,1.576395e+06,9.802001e+05,1.714031e+06,1.990814e+06,2.014875e+06,9.986963e+05,1.093108e+06,1.334691e+06,1.237581e+06,...,1.598919e+06,1.272286e+06,1.024954e+06,1.517614e+06,1.434988e+06,1.253071e+06,1.796903e+06,1.959247e+06,1.377944e+06,1.642902e+06
4,1.947976e+06,2.145146e+06,1.889418e+06,1.756070e+06,1.281853e+06,1.737510e+06,2.758043e+06,2.721496e+06,2.639917e+06,2.665687e+06,...,2.788924e+05,2.495039e+05,4.248775e+05,3.098800e+05,2.888458e+05,4.263454e+05,2.601564e+05,2.230069e+05,2.389870e+05,1.698564e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,2.946200e+07,3.335107e+07,3.520060e+07,2.942578e+07,3.004647e+07,3.151865e+07,3.869054e+07,3.642163e+07,3.396938e+07,3.765264e+07,...,3.547148e+07,3.244064e+07,3.301073e+07,3.187038e+07,3.838530e+07,2.989014e+07,3.527329e+07,2.920353e+07,3.044284e+07,3.695492e+07
78,3.533964e+06,4.744814e+06,3.217944e+06,3.319008e+06,4.928576e+06,3.432516e+06,5.062720e+06,5.280203e+06,4.084184e+06,4.841419e+06,...,2.779232e+06,3.833238e+06,4.436233e+06,4.582209e+06,4.422106e+06,3.285745e+06,4.329191e+06,3.609709e+06,4.725286e+06,3.467679e+06
79,2.096873e+06,2.334722e+06,2.275303e+06,1.700928e+06,2.053808e+06,2.442797e+06,2.437916e+06,1.998224e+06,1.929595e+06,1.900599e+06,...,2.267375e+06,2.866012e+06,2.191374e+06,1.940903e+06,2.732096e+06,1.846165e+06,2.074487e+06,2.443682e+06,1.844960e+06,3.324116e+06
80,1.193444e+06,4.534499e+06,5.824158e+06,4.594740e+06,8.490425e+05,3.976399e+06,4.301930e+06,5.632961e+06,2.697166e+06,1.288474e+06,...,4.308660e+06,1.364190e+06,1.602308e+06,3.487020e+06,3.755398e+06,4.040052e+06,1.644414e+06,3.098998e+06,2.836007e+06,1.862349e+06


In [7]:
df_join_raw_intensity.info()

<class 'pandas.core.frame.DataFrame'>
Index: 82 entries, 0 to 81
Data columns (total 36 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   SecoAmazonas_1.1     82 non-null     float64
 1   SecoAmazonas_1.2     82 non-null     float64
 2   SecoAmazonas_1.3     82 non-null     float64
 3   SecoAmazonas_2.1     82 non-null     float64
 4   SecoAmazonas_2.2     82 non-null     float64
 5   SecoAmazonas_2.3     82 non-null     float64
 6   SecoCusco_1.1        82 non-null     float64
 7   SecoCusco_1.2        82 non-null     float64
 8   SecoCusco_1.3        82 non-null     float64
 9   SecoCusco_2.1        82 non-null     float64
 10  SecoCusco_2.2        82 non-null     float64
 11  SecoCusco_2.3        82 non-null     float64
 12  SecoSanMartin_1.1    82 non-null     float64
 13  SecoSanMartin_1.2    82 non-null     float64
 14  SecoSanMartin_1.3    82 non-null     float64
 15  SecoSanMartin_2.1    82 non-null     float64
 1

In [8]:
check_dataset(df_join_raw_intensity)

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 0
Count zero:	 0
Count positive:	 2952


### Generate graphs

In [9]:
""" from sklearn import preprocessing

X_scaled = preprocessing.RobustScaler().fit_transform(df_join_raw)

df_join_raw_log = pd.DataFrame(X_scaled, columns=df_join_raw.columns)
df_join_raw_log """

' from sklearn import preprocessing\n\nX_scaled = preprocessing.RobustScaler().fit_transform(df_join_raw)\n\ndf_join_raw_log = pd.DataFrame(X_scaled, columns=df_join_raw.columns)\ndf_join_raw_log '

In [10]:
# Transformation (log10)

if not has_transformation:
	df_join_raw_log = log10_global(df_join_raw_intensity)
else:
	df_join_raw_log = df_join_raw_intensity.copy()
df_join_raw_log.head()

,SecoAmazonas_1.1,SecoAmazonas_1.2,SecoAmazonas_1.3,SecoAmazonas_2.1,SecoAmazonas_2.2,SecoAmazonas_2.3,SecoCusco_1.1,SecoCusco_1.2,SecoCusco_1.3,SecoCusco_2.1,...,FrescoCusco_1.3,FrescoCusco_2.1,FrescoCusco_2.2,FrescoCusco_2.3,FrescoSanMartin_1.1,FrescoSanMartin_1.2,FrescoSanMartin_1.3,FrescoSanMartin_2.1,FrescoSanMartin_2.2,FrescoSanMartin_2.3
0,6.289584,6.331457,6.276328,6.244542,6.107838,6.239927,6.440601,6.434808,6.421590,6.425809,...,5.445437,5.397077,5.628264,5.491194,5.460666,5.629762,5.415235,5.348318,5.378374,5.230082
1,7.093810,6.380575,6.456671,6.382825,6.566837,6.651090,6.639265,6.541975,6.596284,6.713756,...,6.111160,6.624436,6.692349,7.106154,7.020618,6.695458,7.091122,6.708999,5.937576,7.095707
2,6.709704,6.783390,6.772010,6.884799,7.046179,6.807104,6.769811,6.774326,6.908646,6.874513,...,6.837353,6.715102,6.757990,6.493360,6.670455,7.092441,6.713908,6.827122,6.877381,6.899714
3,6.251607,6.197665,5.991315,6.234019,6.299031,6.304248,5.999433,6.038663,6.125381,6.092574,...,6.203827,6.104585,6.010704,6.181161,6.156848,6.097976,6.254525,6.292089,6.139232,6.215612
4,6.289584,6.331457,6.276328,6.244542,6.107838,6.239927,6.440601,6.434808,6.421590,6.425809,...,5.445437,5.397077,5.628264,5.491194,5.460666,5.629762,5.415235,5.348318,5.378374,5.230082


In [11]:
check_dataset(df_join_raw_log)

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 0
Count zero:	 0
Count positive:	 2952


In [12]:
# split graph in groups and subgroups

""" def split_groups_subgroups(df_join_raw_log, groups_id, subgroups_id, by_group=False):
	list_df_groups_subgroups = []
	for group in groups_id:
		df_aux = df_join_raw_log.filter(like=group)
		list_aux = []
		
		if by_group:
			list_aux.append(df_aux)
		else:
			for subgroup in subgroups_id[group]:
				list_aux.append(df_aux.filter(like="{}_{}.".format(group, subgroup)))
		list_df_groups_subgroups.append(list_aux)
	return list_df_groups_subgroups """

dict_df_groups_subgroups = split_groups_subgroups(df_join_raw_log, groups_id, subgroups_id)
dict_df_groups_subgroups[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,SecoAmazonas_1.1,SecoAmazonas_1.2,SecoAmazonas_1.3
0,6.289584,6.331457,6.276328
1,7.093810,6.380575,6.456671
2,6.709704,6.783390,6.772010
3,6.251607,6.197665,5.991315
4,6.289584,6.331457,6.276328


In [13]:
check_dataset(dict_df_groups_subgroups[groups_id[0]][subgroups_id[groups_id[0]][0]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 0
Count zero:	 0
Count positive:	 246


In [14]:
# Transpose
dict_groups_subgroups_t = transpose_global(dict_df_groups_subgroups)
dict_groups_subgroups_t[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,0,1,2,3,4,5,6,7,8,9,...,72,73,74,75,76,77,78,79,80,81
0,6.289584,7.093810,6.709704,6.251607,6.289584,6.289584,6.564813,6.448829,6.051900,6.111417,...,7.179433,6.521862,7.018688,7.469262,6.847196,7.469262,6.548262,6.321572,6.076802,5.993872
1,6.331457,6.380575,6.783390,6.197665,6.331457,6.331457,6.599828,6.460548,6.072640,6.070283,...,7.128057,6.500781,7.001935,7.523110,6.666857,7.523110,6.676219,6.368235,6.656529,6.086815
2,6.276328,6.456671,6.772010,5.991315,6.276328,6.276328,6.587890,6.440737,6.090086,6.162088,...,7.090346,6.641258,7.011273,7.546550,6.869051,7.546550,6.507579,6.357039,6.765233,5.877415


In [15]:
check_dataset(dict_groups_subgroups_t[groups_id[0]][subgroups_id[groups_id[0]][0]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 0
Count zero:	 0
Count positive:	 246


In [16]:
# Correlation matrix

dict_groups_subgroups_t_corr = correlation_global(exp, dict_groups_subgroups_t)
dict_groups_subgroups_t_corr[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

SecoAmazonas 1 (3, 82)
SecoAmazonas 2 (3, 82)
SecoCusco 1 (3, 82)
SecoCusco 2 (3, 82)
SecoSanMartin 1 (3, 82)
SecoSanMartin 2 (3, 82)
FrescoAmazonas 1 (3, 82)
FrescoAmazonas 2 (3, 82)
FrescoCusco 1 (3, 82)
FrescoCusco 2 (3, 82)
FrescoSanMartin 1 (3, 82)
FrescoSanMartin 2 (3, 82)


,0,1,2,3,4,5,6,7,8,9,...,72,73,74,75,76,77,78,79,80,81
0,1.000000,0.983523,-0.984848,0.706456,-1.000000,-1.000000,-0.989467,-0.998640,-0.944973,0.992367,...,0.950493,0.859696,0.993732,-0.964331,0.999606,-0.964331,-1.000000,-0.987075,-0.974056,-0.992031
1,0.983523,1.000000,0.999972,-0.822766,0.983523,0.983523,0.999333,0.972762,0.988545,-0.953723,...,-0.991009,-0.753186,-0.997568,0.996295,-0.988211,0.996295,0.983487,0.999783,0.998919,0.952909
2,-0.984848,0.999972,1.000000,0.818489,-0.984848,-0.984848,-0.999579,-0.974470,-0.987388,0.955946,...,0.989980,0.758088,0.998061,-0.995623,0.989329,-0.995623,-0.984814,-0.999911,-0.998543,-0.955151
3,0.706456,-0.822766,0.818489,1.000000,0.706456,0.706456,0.801469,0.668601,0.899123,-0.613786,...,-0.891415,-0.245810,-0.781145,0.868599,-0.726047,0.868599,0.706316,0.810749,0.848299,0.611655
4,-1.000000,0.983523,-0.984848,0.706456,1.000000,-1.000000,-0.989467,-0.998640,-0.944973,0.992367,...,0.950493,0.859696,0.993732,-0.964331,0.999606,-0.964331,-1.000000,-0.987075,-0.974056,-0.992031


In [17]:
# Check correlation matrices

dict_groups_subgroups_t_corr

{'SecoAmazonas': {'1':           0         1         2         3         4         5         6   \
  0   1.000000  0.983523 -0.984848  0.706456 -1.000000 -1.000000 -0.989467   
  1   0.983523  1.000000  0.999972 -0.822766  0.983523  0.983523  0.999333   
  2  -0.984848  0.999972  1.000000  0.818489 -0.984848 -0.984848 -0.999579   
  3   0.706456 -0.822766  0.818489  1.000000  0.706456  0.706456  0.801469   
  4  -1.000000  0.983523 -0.984848  0.706456  1.000000 -1.000000 -0.989467   
  ..       ...       ...       ...       ...       ...       ...       ...   
  77 -0.964331  0.996295 -0.995623  0.868599 -0.964331 -0.964331 -0.992491   
  78 -1.000000  0.983487 -0.984814  0.706316 -1.000000 -1.000000 -0.989438   
  79 -0.987075  0.999783 -0.999911  0.810749 -0.987075 -0.987075 -0.999877   
  80 -0.974056  0.998919 -0.998543  0.848299 -0.974056 -0.974056 -0.996556   
  81 -0.992031  0.952909 -0.955151  0.611655 -0.992031 -0.992031 -0.963344   
  
            7         8         9   ... 

In [18]:
check_dataset(dict_groups_subgroups_t_corr[groups_id[0]][subgroups_id[groups_id[0]][1]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 3682
Count zero:	 0
Count positive:	 3042


In [19]:
# Build graph (corpus graphs)

dict_groups_subgroups_t_corr_g = build_graph_weight_global_directed(exp, dict_groups_subgroups_t_corr, threshold=threshold_corr)
dict_groups_subgroups_t_corr_g[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,source,target,weight,subgroup
0,0,1,0.983523,1
1,0,2,-0.984848,1
2,0,3,0.706456,1
3,0,4,-1.000000,1
4,0,5,-1.000000,1


In [20]:
def create_graph_data_directed_features(exp, groups_id, subgroups_id, dict_df_groups_subgroups, df_join_raw_metadata):	
	for group_id in tqdm(groups_id):
		for subgroup_id in tqdm(subgroups_id[group_id]):
			df_weighted_edges = pd.read_csv("experiments/output/{}/preprocessing/edges/{}_{}.csv".format(exp, group_id, subgroup_id))
			# print(df_weighted_edges)
			G = nx.from_pandas_edgelist(df_weighted_edges, "source", "target", edge_attr=["weight", "subgroup"])
			dict_id_idx = dict(zip(list(G.nodes()), range(G.number_of_nodes())))
			G = nx.relabel_nodes(G, dict_id_idx)

			df_nodes = dict_df_groups_subgroups[group_id][subgroup_id].loc[list(dict_id_idx.keys())] # A_1.1, A_1.2, A_1.3
			# from IPython.display import display
			# display(df_nodes)

			# nodes, with node features
			metadata = df_join_raw_metadata.loc[df_nodes.index] # Average Rt, Average Mz
			# intensity = df_join_raw_log.loc[df_nodes.index] # A_1.1, A_1.2, A_1.3, A_2.1, ...

			""" e = 1e-8
			mz = metadata.iloc[:, 1].values
			rt = metadata.iloc[:, 0].values
			intensity_mean = df_nodes.mean(axis=1).values
			intensity_std = df_nodes.std(axis=1).values
			intensity_cv = intensity_std / intensity_mean
			presence_ratio = (df_nodes > 0).mean(axis=1)

			mz_log = np.log10(mz + e)
			# intensity_mean_log = np.log10(intensity_mean + e)
			
			# mz_z = (mz_log - mz_log.mean()) / mz_log.std() # z-score
			rt_z = (rt - rt.mean()) / rt.std() # z-score
			# intensity_mean_z = (intensity_mean_log - intensity_mean_log.mean()) / intensity_mean_log.std() # z-score

			data_node = {
				"idx": list(dict_id_idx.values()),
				"id": list(dict_id_idx.keys()),
				"mz": mz_log,
				"rt": rt_z,
				"intensity_mean": intensity_mean,
				"intensity_std": intensity_std,
				"intensity_cv": intensity_cv,
				"presence_ratio": presence_ratio
			}
			for i in range(len(df_nodes.columns)):
				data_node[i] = df_nodes.iloc[:, i]

			df_node_features = pd.DataFrame(data_node)
			# df_node_features.insert(0, "idx", list(dict_id_idx.values()))
			# df_node_features.insert(1, "id", list(dict_id_idx.keys()))
			df_node_features.to_csv("experiments/output/{}/preprocessing/graphs_data/nodes_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# print(df_node_features) """

			data_node = {
				"idx": list(dict_id_idx.values()),
				"id": list(dict_id_idx.keys()),
				"mz": metadata.iloc[:, 1].values,
				"rt": metadata.iloc[:, 0].values,
			}
			for i in range(len(df_nodes.columns)):
				data_node[i] = df_nodes.iloc[:, i]

			df_node_features = pd.DataFrame(data_node)
			df_node_features.to_csv("experiments/output/{}/preprocessing/graphs_data/nodes_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# print(df_node_features)

			# edges
			edges = list(G.edges())
			df_edges = pd.DataFrame(edges, columns=["source", "target"])
			df_edges["weight"] = [G.get_edge_data(*edge)["weight"] for edge in edges]
			df_edges["subgroup"] = [G.get_edge_data(*edge)["subgroup"] for edge in edges]
			df_edges.to_csv("experiments/output/{}/preprocessing/graphs_data/edges_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)


In [21]:
# create dataset - nodes/edge data for PyTorch Geometric/DGL framework

# IMPORTANT
dict_df_groups_subgroups_ = split_groups_subgroups(df_join_raw_intensity, groups_id, subgroups_id) # Important (intesities without Log)

for data_variation in data_variations:
	if data_variation == "none":
		create_graph_data_directed_features(exp, groups_id, subgroups_id, dict_df_groups_subgroups_, df_join_raw_metadata)
	else:
		# dynamic graph to static graph
		create_graph_data_directed_variation(exp, groups_id, subgroups_id, dict_df_groups_subgroups_, data_variation)

100%|██████████| 6/6 [00:00<00:00, 36.25it/s]


In [22]:
# details
list_details = []
	
for group_id in groups_id:
	subgroups_id_ = []
	for data_variation in data_variations:
		if data_variation == "none":
			subgroups_id_ += subgroups_id[group_id]
		else:
			subgroups_id_ += [data_variation]
	# print(subgroups)
	
	for subgroup_id_ in subgroups_id_:
		df_edges = pd.read_csv("experiments/output/{}/preprocessing/graphs_data/edges_{}_{}.csv".format(exp, group_id, subgroup_id_))

		G = nx.from_pandas_edgelist(df_edges.iloc[:, [0, 1]])
		list_details.append([group_id, subgroup_id_, G.number_of_nodes(), G.number_of_edges(), nx.density(G), np.nan, nx.is_connected(G)])

df_details = pd.DataFrame(list_details, columns=["Group", "Subgroup", "Num. nodes", "Num. edges", "Density", "Diameter", "Is connected"])
df_details.to_csv("experiments/output/{}/preprocessing/graphs_data/summary.csv".format(exp), index=False)

df_details = pd.read_csv("experiments/output/{}/preprocessing/graphs_data/summary.csv".format(exp))
df_details

,Group,Subgroup,Num. nodes,Num. edges,Density,Diameter,Is connected
0,SecoAmazonas,1,82,2801,0.843421,NaN,True
1,SecoAmazonas,2,82,2538,0.764228,NaN,True
2,SecoCusco,1,82,2961,0.891599,NaN,True
3,SecoCusco,2,82,2666,0.802770,NaN,True
4,SecoSanMartin,1,82,2636,0.793737,NaN,True
5,SecoSanMartin,2,82,2257,0.679615,NaN,True
6,FrescoAmazonas,1,82,2790,0.840108,NaN,True
7,FrescoAmazonas,2,82,2488,0.749172,NaN,True
8,FrescoCusco,1,82,2305,0.694068,NaN,True
9,FrescoCusco,2,82,2365,0.712135,NaN,True
